# Extended Thinking Latency: Claude Opus 4.7

This notebook measures the impact of **extended thinking** on latency for
Claude Opus 4.7 across two Amazon Bedrock streaming API paths:

| API | Endpoint class | Protocol | Bedrock endpoint |
|-----|---------------|----------|------------------|
| **Converse** | `BedrockConverseStream` | AWS event-stream | `bedrock-runtime` |
| **Anthropic Messages** | `AnthropicMessagesStream` | SSE (server-sent events) | `bedrock-mantle` |

For each API path we compare three configurations:

1. **No thinking** — `thinking: {"type": "disabled"}` (baseline)
2. **Adaptive thinking** — `thinking: {"type": "adaptive"}` (recommended for Opus 4.7)
3. **Adaptive thinking, display omitted** — same as above with `display: "omitted"`
   (skips streaming thinking tokens, reducing time-to-first-visible-text)

We measure two TTFT variants:

- **TTFT (visible)** — time to first *visible text* token (`ttft_visible_tokens_only=True`)
- **TTFT (any)** — time to first token of *any* kind, including thinking deltas
  (`ttft_visible_tokens_only=False`)

### Key expectations

- Thinking adds latency to TTFT (visible) because the model reasons before producing text
- `display: "omitted"` should reduce TTFT (visible) compared to `display: "summarized"`
  because no thinking tokens are streamed
- TTLT increases with thinking because more output tokens are generated
- TTFT (any) captures when the model first starts producing output, which is earlier
  than TTFT (visible) when thinking is enabled

In [ ]:
# %pip install "llmeter[anthropic,plotting]<1"

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Anti-caching callback

Prepends a unique prefix to each request to defeat prompt-level caching.

In [2]:
import uuid
from llmeter.callbacks import Callback


class NoCacheCallback(Callback):
    """Prepend a unique prefix to each request to defeat prompt caching."""

    def __init__(self):
        self._counter = 0

    async def before_invoke(self, payload: dict) -> None:
        self._counter += 1
        tag = f" [req-{self._counter:04d}-{uuid.uuid4().hex[:5]}]"

        messages = payload.get("messages", [])
        if not messages:
            return

        last_msg = messages[-1]
        content = last_msg.get("content")

        # Messages API format: content is a plain string
        if isinstance(content, str):
            last_msg["content"] = tag + content
            return

        # Converse format: content is a list of blocks
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and "text" in block:
                    block["text"] = tag + block["text"]
                    return

## Configuration

In [3]:
REGION = "us-east-1"
N_REQUESTS = 10
OUTPUT_PATH = "outputs/opus-4.7-extended-thinking"

# A prompt that benefits from reasoning
PROMPT = (
    "A farmer has 3 fields. The first field produces 40% more wheat than the second. "
    "The third field produces 25% less than the first. Together they produce 8,200 kg. "
    "How much does each field produce? Show your work step by step."
)
MAX_TOKENS = 4096

## Configure endpoints

We create **four** streaming endpoints — two per API path, one measuring
TTFT to visible text only (default) and one measuring TTFT to any token
including thinking.

In [4]:
from llmeter.endpoints.bedrock import BedrockConverseStream
from llmeter.endpoints.anthropic_messages import AnthropicMessagesStream

# --- Converse API (bedrock-runtime) ---
converse_visible = BedrockConverseStream(
    model_id="global.anthropic.claude-opus-4-7",
    region=REGION,
    endpoint_name="converse (visible TTFT)",
    ttft_visible_tokens_only=True,
)

converse_any = BedrockConverseStream(
    model_id="global.anthropic.claude-opus-4-7",
    region=REGION,
    endpoint_name="converse (any TTFT)",
    ttft_visible_tokens_only=False,
)

# --- Anthropic Messages API (bedrock-mantle) ---
messages_visible = AnthropicMessagesStream(
    model_id="anthropic.claude-opus-4-7",
    provider="bedrock-mantle",
    aws_region=REGION,
    endpoint_name="messages (visible TTFT)",
    ttft_visible_tokens_only=True,
)

messages_any = AnthropicMessagesStream(
    model_id="anthropic.claude-opus-4-7",
    provider="bedrock-mantle",
    aws_region=REGION,
    endpoint_name="messages (any TTFT)",
    ttft_visible_tokens_only=False,
)

## Build payloads

Three thinking configurations per API path.

Note: Opus 4.7 only supports adaptive thinking (`type: "adaptive"`).
Manual `type: "enabled"` with `budget_tokens` returns a 400 error on this model.
The `display` field defaults to `"omitted"` on Opus 4.7, so we set
`"summarized"` explicitly for the summarized variant.

In [5]:
from llmeter.endpoints.bedrock import BedrockBase
from llmeter.endpoints.anthropic_messages import AnthropicMessagesEndpoint

# --- Converse payloads ---
# Thinking config goes in additionalModelRequestFields for Converse
converse_no_thinking = BedrockBase.create_payload(
    PROMPT,
    max_tokens=MAX_TOKENS,
    additionalModelRequestFields={"thinking": {"type": "disabled"}},
)

converse_thinking_summarized = BedrockBase.create_payload(
    PROMPT,
    max_tokens=MAX_TOKENS,
    additionalModelRequestFields={
        "thinking": {"type": "adaptive", "display": "summarized"}
    },
)

converse_thinking_omitted = BedrockBase.create_payload(
    PROMPT,
    max_tokens=MAX_TOKENS,
    additionalModelRequestFields={
        "thinking": {"type": "adaptive", "display": "omitted"}
    },
)

# --- Messages API payloads ---
messages_no_thinking = AnthropicMessagesEndpoint.create_payload(
    PROMPT,
    max_tokens=MAX_TOKENS,
    thinking={"type": "disabled"},
)

messages_thinking_summarized = AnthropicMessagesEndpoint.create_payload(
    PROMPT,
    max_tokens=MAX_TOKENS,
    thinking={"type": "adaptive", "display": "summarized"},
)

messages_thinking_omitted = AnthropicMessagesEndpoint.create_payload(
    PROMPT,
    max_tokens=MAX_TOKENS,
    thinking={"type": "adaptive", "display": "omitted"},
)

## Verify endpoints

Quick smoke test — one request per configuration to confirm everything works.

In [6]:
test_cases = [
    ("Converse / no thinking", converse_visible, converse_no_thinking),
    ("Converse / adaptive (summarized)", converse_visible, converse_thinking_summarized),
    ("Converse / adaptive (omitted)", converse_visible, converse_thinking_omitted),
    ("Messages / no thinking", messages_visible, messages_no_thinking),
    ("Messages / adaptive (summarized)", messages_visible, messages_thinking_summarized),
    ("Messages / adaptive (omitted)", messages_visible, messages_thinking_omitted),
]

for name, ep, payload in test_cases:
    r = ep.invoke(payload)
    assert r.error is None, f"{name} error: {r.error}"
    print(
        f"{name:>40}  "
        f"TTFT={r.time_to_first_token:.3f}s  "
        f"TTLT={r.time_to_last_token:.3f}s  "
        f"out={r.num_tokens_output}"
    )

                  Converse / no thinking  TTFT=1.689s  TTLT=6.068s  out=622
        Converse / adaptive (summarized)  TTFT=4.324s  TTLT=6.692s  out=777
           Converse / adaptive (omitted)  TTFT=26.235s  TTLT=29.358s  out=737
                  Messages / no thinking  TTFT=2.021s  TTLT=7.027s  out=629
        Messages / adaptive (summarized)  TTFT=3.379s  TTLT=6.212s  out=801
           Messages / adaptive (omitted)  TTFT=2.415s  TTLT=5.759s  out=730


## Run the benchmark

We run each configuration through `Runner` to collect enough samples for
meaningful statistics. Each run uses a fresh `NoCacheCallback` to prevent
prompt caching.

In [7]:
from llmeter.runner import Runner

results = {}

runs = [
    # (label, endpoint, payload)
    ("converse-no-thinking", converse_visible, converse_no_thinking),
    ("converse-thinking-summarized", converse_visible, converse_thinking_summarized),
    ("converse-thinking-omitted", converse_visible, converse_thinking_omitted),
    ("converse-thinking-any-ttft", converse_any, converse_thinking_summarized),
    ("messages-no-thinking", messages_visible, messages_no_thinking),
    ("messages-thinking-summarized", messages_visible, messages_thinking_summarized),
    ("messages-thinking-omitted", messages_visible, messages_thinking_omitted),
    ("messages-thinking-any-ttft", messages_any, messages_thinking_summarized),
]

for label, endpoint, payload in runs:
    print(f"\n--- {label} ---")
    runner = Runner(
        endpoint=endpoint,
        output_path=OUTPUT_PATH,
        callbacks=[NoCacheCallback()],
    )
    result = await runner.run(
        payload=payload,
        n_requests=N_REQUESTS,
        clients=1,
        run_name=label,
    )
    results[label] = result
    print(
        f"  {result.total_requests} requests  "
        f"p50 TTFT={result.stats['time_to_first_token-p50']:.3f}s  "
        f"p50 TTLT={result.stats['time_to_last_token-p50']:.3f}s"
    )


--- converse-no-thinking ---


Total requests:   0%|          | 0/10 [00:00<?, ?it/s]

Throughput,TTFT,TTLT,Tokens,Errors
rpm 6.7,p50_ttft 1.971s,p50_ttlt 7.738s,input_tokens 909.0,fail 0.0
output_tps 65.9 tok/s,p90_ttft 20.364s,p90_ttlt 25.973s,output_tokens 6671.0,
p50_tps 122.7 tok/s,,,,


  10 requests  p50 TTFT=1.971s  p50 TTLT=7.738s

--- converse-thinking-summarized ---


Total requests:   0%|          | 0/10 [00:00<?, ?it/s]

...

  10 requests  p50 TTFT=3.077s  p50 TTLT=6.734s

--- messages-thinking-any-ttft ---


Total requests:   0%|          | 0/10 [00:00<?, ?it/s]

Throughput,TTFT,TTLT,Tokens,Errors
rpm 5.0,p50_ttft 2.447s,p50_ttlt 6.760s,input_tokens 815.0,fail 0.0
output_tps 63.6 tok/s,p90_ttft 29.852s,p90_ttlt 36.524s,output_tokens 7288.0,
p50_tps 211.7 tok/s,,,,


Timeout waiting for response


  9 requests  p50 TTFT=2.447s  p50 TTLT=6.760s


## Summary table

In [8]:
import pandas as pd

rows = []
for label, result in results.items():
    s = result.stats
    rows.append({
        "Configuration": label,
        "p50 TTFT (s)": round(s["time_to_first_token-p50"], 3),
        "p90 TTFT (s)": round(s["time_to_first_token-p90"], 3),
        "p50 TTLT (s)": round(s["time_to_last_token-p50"], 3),
        "p90 TTLT (s)": round(s["time_to_last_token-p90"], 3),
        "Avg output tokens": round(s.get("num_tokens_output-mean", 0)),
        "Requests": result.total_requests,
        "Errors": s.get("errors", 0),
    })

df = pd.DataFrame(rows).set_index("Configuration")
df

,p50 TTFT (s),p90 TTFT (s),p50 TTLT (s),p90 TTLT (s),Avg output tokens,Requests,Errors
Configuration,,,,,,,
converse-no-thinking,1.971,20.364,7.738,25.973,0,10,0
converse-thinking-summarized,4.048,31.044,6.824,34.086,0,10,0
converse-thinking-omitted,3.600,25.766,7.531,28.689,0,10,0
converse-thinking-any-ttft,3.082,6.276,6.594,9.963,0,10,0
messages-no-thinking,1.109,24.495,5.697,30.082,0,10,0
messages-thinking-summarized,7.132,26.860,10.248,30.793,0,10,0
messages-thinking-omitted,3.077,22.238,6.734,24.422,0,10,0
messages-thinking-any-ttft,2.447,29.852,6.760,36.524,0,9,0


## Plot: TTFT by configuration

Box plots showing the TTFT distribution for each configuration, grouped by
API path.

In [9]:
import plotly.graph_objects as go
from llmeter.plotting import boxplot_by_dimension

colors = {
    "no-thinking": "#4CAF50",
    "thinking-summarized": "#2196F3",
    "thinking-omitted": "#FF9800",
    "thinking-any-ttft": "#9C27B0",
}


def color_for(label):
    for key, color in colors.items():
        if key in label:
            return color
    return "#607D8B"


fig = go.Figure()
for label, result in results.items():
    fig.add_trace(
        boxplot_by_dimension(
            result,
            "time_to_first_token",
            name=label,
            marker=dict(color=color_for(label)),
            boxpoints="all",
            jitter=0.3,
        )
    )

fig.update_layout(
    title="TTFT by Thinking Configuration (Opus 4.7, us-east-1)",
    yaxis_title="Time to First Token (seconds)",
    showlegend=False,
    height=500,
    template="plotly_white",
)
fig.show()

## Plot: TTLT by configuration

Total response time including all output tokens (thinking + text).

In [10]:
fig_ttlt = go.Figure()
for label, result in results.items():
    fig_ttlt.add_trace(
        boxplot_by_dimension(
            result,
            "time_to_last_token",
            name=label,
            marker=dict(color=color_for(label)),
            boxpoints="all",
            jitter=0.3,
        )
    )

fig_ttlt.update_layout(
    title="TTLT by Thinking Configuration (Opus 4.7, us-east-1)",
    yaxis_title="Time to Last Token (seconds)",
    showlegend=False,
    height=500,
    template="plotly_white",
)
fig_ttlt.show()

## Plot: TTFT visible vs any (thinking enabled)

Direct comparison of the two TTFT measurement modes for the same thinking
configuration. This shows the gap between when the model starts thinking
and when the user sees the first visible text.

In [11]:
from llmeter.plotting import histogram_by_dimension

for api in ["converse", "messages"]:
    visible_key = f"{api}-thinking-summarized"
    any_key = f"{api}-thinking-any-ttft"

    if visible_key not in results or any_key not in results:
        continue

    fig_cmp = go.Figure()
    fig_cmp.add_trace(
        histogram_by_dimension(
            results[any_key],
            "time_to_first_token",
            name="TTFT (any token)",
            opacity=0.7,
            marker_color="#9C27B0",
        )
    )
    fig_cmp.add_trace(
        histogram_by_dimension(
            results[visible_key],
            "time_to_first_token",
            name="TTFT (visible text)",
            opacity=0.7,
            marker_color="#2196F3",
        )
    )
    fig_cmp.update_layout(
        title=f"TTFT: Any Token vs Visible Text ({api.title()}, adaptive thinking)",
        xaxis_title="Time to First Token (seconds)",
        yaxis_title="Probability",
        barmode="overlay",
        height=400,
        template="plotly_white",
    )
    fig_cmp.show()

## Observations

Expected patterns:

- **No thinking** has the lowest TTFT and TTLT — the model produces text immediately
- **Adaptive thinking (summarized)** increases TTFT (visible) because thinking
  tokens stream before the text. TTFT (any) is lower because it captures the
  first thinking delta
- **Adaptive thinking (omitted)** should have TTFT (visible) between the other
  two — no thinking tokens are streamed, so text starts sooner than summarized
  mode, but the model still reasons internally before producing text
- **TTLT** increases with thinking because `output_tokens` includes both
  thinking and text tokens
- The gap between TTFT (any) and TTFT (visible) represents the time the model
  spends reasoning before producing visible output

### Token accounting note

The Anthropic API reports a single `output_tokens` count that includes both
thinking and visible text tokens. There is no separate `reasoning_tokens`
field. This means `num_tokens_output` in the results reflects total billed
output, and `num_tokens_output_reasoning` is `None`.

To get more statistically significant results, increase `N_REQUESTS` or use
`run_duration` for time-bound runs.